# 🎓 NLP Workshop: Fine-Tuning with Hugging Face

**Goal:** By the end of this session, you will fine-tune a sentiment classifier using a pretrained transformer.

## Agenda
1. **Instructor Demo:** We will load a dataset, tokenize it, and fine-tune BERT (15 mins).
2. **Your Turn:** You will modify the model architecture and preprocessing steps (20 mins).
3. **Mini Challenge:** Improve the model's speed or accuracy (5 mins).
4. **Debrief:** Reflection (5 mins).

## 🛠️ Setup
Let's install the necessary libraries.

In [1]:
%pip install transformers datasets evaluate accelerate torch scikit-learn

  Using cached tokenizers-0.22.2-cp39-abi3-macosx_11_0_arm64.whl.metadata (7.3 kB)
  Using cached safetensors-0.7.0-cp38-abi3-macosx_11_0_arm64.whl.metadata (4.1 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.4/10.4 MB 9.6 MB/s  0:00:01 eta 0:00:01
Using cached tokenizers-0.22.2-cp39-abi3-macosx_11_0_arm64.whl (3.0 MB)
Using cached safetensors-0.7.0-cp38-abi3-macosx_11_0_arm64.whl (447 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5/5 [evaluate]3/5 [transformers]
Note: you may need to restart the kernel to use updated packages.


## Phase 1: Instructor Demo (Watch & Learn)

We are going to take the **IMDB dataset** (movie reviews) and train a model to predict if a review is **Positive (1)** or **Negative (0)**.

### Step 1: Load and View Data

In [2]:
from datasets import load_dataset

# 1. Load the dataset
dataset = load_dataset("imdb")

# OPTIMIZATION FOR CLASS: Create a small subset so training finishes in < 2 mins
# We are taking 1000 training examples and 200 test examples
small_train_dataset = dataset["train"].shuffle(seed=42).select(range(1000))
small_eval_dataset = dataset["test"].shuffle(seed=42).select(range(200))

# Let's look at one example
print(small_train_dataset[0])

/Users/gauravmapari/COEP/.venv/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Generating unsupervised split: 100%|██████████| 50000/50000 [00:00<00:00, 844468.07 examples/s]

{'text': 'There is no relation at all between Fortier and Profiler but the fact that both are police series about violent crimes. Profiler looks crispy, Fortier looks classic. Profiler plots are quite simple. Fortier\'s plot are far more complicated... Fortier looks more like Prime Suspect, if we have to spot similarities... The main character is weak and weirdo, but have "clairvoyance". People like to compare, to judge, to evaluate. How about just enjoying? Funny thing too, people writing Fortier looks American but, on the other hand, arguing they prefer American series (!!!). Maybe it\'s the language, or the spirit, but I think this series is more English than American. By the way, the actors are really good and funny. The acting is not superficial at all...', 'label': 1}


### Step 2: Preprocessing (Tokenization)
Computers don't understand text; they understand numbers. We use a **Tokenizer** to convert text into Input IDs.

In [3]:
from transformers import AutoTokenizer

# Define the checkpoint we want to use (Classic BERT)
model_checkpoint = "bert-base-uncased"

# Load tokenizer
tokenizer = AutoTokenizer.from_pretrained(model_checkpoint)

# Function to tokenize the data
def tokenize_function(examples):
    # Truncation ensures no input exceeds the model's limit
    return tokenizer(examples["text"], padding="max_length", truncation=True, max_length=128)

# Apply tokenization to the whole dataset
tokenized_train = small_train_dataset.map(tokenize_function, batched=True)
tokenized_eval = small_eval_dataset.map(tokenize_function, batched=True)

Map: 100%|██████████| 200/200 [00:00<00:00, 9776.70 examples/s]


### Step 3: Load the Model & Metrics
We load a pre-trained BERT model, but we tell it we have **2 labels** (Positive/Negative).

In [4]:
from transformers import AutoModelForSequenceClassification
import evaluate
import numpy as np

# Load the model
model = AutoModelForSequenceClassification.from_pretrained(model_checkpoint, num_labels=2)

# Setup evaluation metric (Accuracy)
metric = evaluate.load("accuracy")

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    predictions = np.argmax(logits, axis=-1)
    return metric.compute(predictions=predictions, references=labels)

Loading weights: 100%|██████████| 199/199 [00:00<00:00, 2787.51it/s, Materializing param=bert.pooler.dense.weight]                               
BertForSequenceClassification LOAD REPORT from: bert-base-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
classifier.weight                          | MISSING    | 
classifier.bias                            | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those p

### Step 4: Training
We use the `Trainer` API, which handles the training loop (batches, gradients, logging) for us.

In [5]:
from transformers import TrainingArguments, Trainer

# Define hyperparameters
training_args = TrainingArguments(
    output_dir="test_trainer",
    evaluation_strategy="epoch",
    num_train_epochs=1,          # Keeping it short for demo
    learning_rate=2e-5,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
)

# Initialize Trainer
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_train,
    eval_dataset=tokenized_eval,
    compute_metrics=compute_metrics,
)

# Train!
trainer.train()

TypeError: TrainingArguments.__init__() got an unexpected keyword argument 'evaluation_strategy'

### Step 5: Inference (Using the Model)
Let's test it on a fake review.

In [ ]:
text = "This movie was honestly pretty terrible. The acting was wooden."

# Preprocess the text manually
inputs = tokenizer(text, return_tensors="pt").to("cuda" if model.device.type == "cuda" else "cpu")

# Get prediction
outputs = model(**inputs)
prediction = outputs.logits.argmax().item()

print(f"Review: {text}")
print(f"Sentiment: {'Positive' if prediction == 1 else 'Negative'}")

---

## Phase 2: Student Turn (Your Code)

**Instructions:**
1. Run the cells above to ensure your environment works.
2. Complete the exercises below by modifying the code.
3. Work in your breakout groups.

### Exercise 1: Switch to a Faster Model
`bert-base-uncased` is powerful but heavy. Switch the pipeline to use `distilbert-base-uncased`.

In [ ]:
# TODO: Change this checkpoint
student_checkpoint = "bert-base-uncased" # <--- Change this to "distilbert-base-uncased"

# TODO: Re-initialize the tokenizer and model with this new checkpoint
student_tokenizer = AutoTokenizer.from_pretrained(student_checkpoint)
student_model = AutoModelForSequenceClassification.from_pretrained(student_checkpoint, num_labels=2)

print("Model architecture loaded:", student_checkpoint)

### Exercise 2: Change Preprocessing
We used `max_length=128`. In real documents, reviews can be longer. 
1. Change `max_length` to **256**.
2. Re-run the map function.

In [ ]:
def student_tokenize_function(examples):
    # TODO: Modify the max_length parameter below
    return student_tokenizer(examples["text"], padding="max_length", truncation=True, max_length=128)

student_tokenized_train = small_train_dataset.map(student_tokenize_function, batched=True)
student_tokenized_eval = small_eval_dataset.map(student_tokenize_function, batched=True)

print("New shape of input_ids:", len(student_tokenized_train[0]['input_ids']))

### Exercise 3: Freeze Layers (Advanced)
To speed up training, we can "freeze" the base model and only train the classifier head. 
Uncomment the code below to see how it affects training speed.

In [ ]:
# for param in student_model.base_model.parameters():
#     param.requires_grad = False

# print("Base model layers frozen!")

## Phase 3: Mini Challenge 🏆

**Goal:** Improve your validation accuracy or training speed.

Modify `TrainingArguments` below. 
* *Hint:* Try changing `learning_rate` (e.g., 5e-5), `num_train_epochs`, or `per_device_train_batch_size`.
* *Hint:* Did freezing the layers help speed?

In [ ]:
challenge_args = TrainingArguments(
    output_dir="challenge_trainer",
    evaluation_strategy="epoch",
    num_train_epochs=1,          # TODO: Try 2 or 3?
    learning_rate=2e-5,          # TODO: Try 5e-5?
    per_device_train_batch_size=16,
)

challenge_trainer = Trainer(
    model=student_model,
    args=challenge_args,
    train_dataset=student_tokenized_train,
    eval_dataset=student_tokenized_eval,
    compute_metrics=compute_metrics,
)

challenge_trainer.train()

## Phase 4: Debrief 📝

1. **What broke?** Did changing the model to `distilbert` cause any errors with the tokenizer?
2. **What surprised you?** Did increasing `max_length` make training significantly slower?
3. **Results:** Who got the highest accuracy?